# ReMorph — Final Training Colab (Production Run)

**Goal:** run the final staged TRL GRPO training (500 -> 1500 -> 3000 prompts), produce judge-ready charts, and publish artifacts for Space + UI handoff.

**Important:** this notebook is designed for **real training only** (no dry-run path).

**Runtime:** GPU required.

**Repo:** `VedantPancholi/remorph-openenv-submission`.

## 0) Authentication + platform rules

In Colab, add `HF_TOKEN` in **Secrets** (read + write scope for dataset/model uploads).

- Colab cannot directly use Hugging Face Space hardware.
- From Colab you can still launch HF Jobs (A10G/A100) with the same token.

In [ ]:
import os
import json
import subprocess

try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab userdata")
except Exception:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in environment/secret"
    print("HF_TOKEN loaded from environment")

print("HF token present:", bool(os.environ.get("HF_TOKEN")))

## 1) Runtime and dependency preflight

In [ ]:
!nvidia-smi || true

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU runtime is required for this final training notebook.")

## 2) Clone + lock dependencies (production-safe)

This cell hard-pins key TRL dependencies and verifies them before training.

If install errors occur, restart session and rerun from cell 2 onward.

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/VedantPancholi/remorph-openenv-submission.git}"
REPO_DIR="${REPO_DIR:-remorph-openenv-submission}"
if [[ ! -d "$REPO_DIR" ]]; then git clone --depth 1 "$REPO_URL" "$REPO_DIR"; fi
cd "$REPO_DIR"
git pull --ff-only || true

pip install -q -U pip setuptools wheel
pip install -q -r requirements.txt
pip install -q -r requirements-training.txt -c pip_constraints.txt
pip install -q --upgrade --force-reinstall --no-cache-dir "transformers>=5.2.0,<6"
pip install -q "jmespath>=1.0.0,<2" "pandas>=2.0,<3"

python - <<'PY'
import transformers, jmespath
print('transformers', transformers.__version__)
print('jmespath', jmespath.__version__)
ver = tuple(int(x) for x in transformers.__version__.split('+')[0].split('.')[:3])
assert ver >= (5,2,0), f"transformers too old: {transformers.__version__}"
print('dependency preflight passed')
PY

python scripts/smoke_grpo_trainer_deps.py --model "${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
echo "Ready in $(pwd)"

## 3) Final staged training run (no dry-run)

Default here is **FULL** profile for submission-grade output.

Tunable knobs:

- `RUN_PROFILE=full|quick`
- `MODEL` (default `Qwen/Qwen2.5-0.5B-Instruct`)
- `MAX_STEPS_V1|V2|V3`
- `HF_DATASET_REPO`
- `UPLOAD_PATH_IN_REPO`
- `SKIP_UPLOAD=0|1`

For final submission, keep `SKIP_UPLOAD=0` and unique `UPLOAD_PATH_IN_REPO` per run.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission

export RUN_PROFILE="${RUN_PROFILE:-full}"      # final notebook default
export MODEL="${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
export MAX_STEPS_V1="${MAX_STEPS_V1:-100}"
export MAX_STEPS_V2="${MAX_STEPS_V2:-200}"
export MAX_STEPS_V3="${MAX_STEPS_V3:-300}"
export HF_DATASET_REPO="${HF_DATASET_REPO:-Jenish31/remorph-training-artifacts}"
export UPLOAD_PATH_IN_REPO="${UPLOAD_PATH_IN_REPO:-colab_final_${RUN_PROFILE}_run}"
export SKIP_UPLOAD="${SKIP_UPLOAD:-0}"

echo "RUN_PROFILE=$RUN_PROFILE"
echo "MODEL=$MODEL"
echo "STEPS=($MAX_STEPS_V1,$MAX_STEPS_V2,$MAX_STEPS_V3)"
echo "UPLOAD_PATH_IN_REPO=$UPLOAD_PATH_IN_REPO"

if [ "$RUN_PROFILE" = "quick" ]; then
  chmod +x scripts/hf_run_quick_confidence.sh
  bash scripts/hf_run_quick_confidence.sh
else
  chmod +x scripts/hf_run_staged_grpo_full.sh
  bash scripts/hf_run_staged_grpo_full.sh
fi

## 4) Metrics dashboard + evidence bundle

This section renders summary tables/charts directly in Colab and then generates handoff artifacts:

- Space checks
- SQLite bridge
- Deterministic UI payload
- Best-run promotion manifest

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

repo = Path("remorph-openenv-submission")
sub = repo / "artifacts" / "submission"

summary_paths = sorted(sub.glob("trl_grpo_run_v*/trl_grpo_training_summary.json"))
if not summary_paths:
    summary_paths = sorted(sub.glob("trl_grpo_run_quick_v*/trl_grpo_training_summary.json"))

rows = []
for p in summary_paths:
    s = json.loads(p.read_text())
    metrics_rows = s.get("metrics_rows", [])
    eval_vals = [r.get("eval/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("eval/reward"), (int, float))]
    reward_vals = [r.get("train/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/reward"), (int, float))]
    frac_vals = [r.get("train/frac_reward_zero_std") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/frac_reward_zero_std"), (int, float))]
    rows.append({
        "run": p.parent.name,
        "model": s.get("model_name"),
        "status": s.get("status"),
        "reward_best": max(reward_vals) if reward_vals else None,
        "eval_reward_best": max(eval_vals) if eval_vals else None,
        "frac_reward_zero_std_last": frac_vals[-1] if frac_vals else None,
        "steps_logged": len(metrics_rows),
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No GRPO summary files found yet.")
else:
    display(df)
    chart_df = df.dropna(subset=["eval_reward_best"])
    if not chart_df.empty:
        plt.figure(figsize=(8,4))
        plt.bar(chart_df["run"], chart_df["eval_reward_best"])
        plt.title("Best eval reward by run")
        plt.ylabel("eval_reward_best")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()

!cd remorph-openenv-submission && python scripts/space_submission_checks.py
!cd remorph-openenv-submission && python scripts/sqlite_artifact_bridge.py
!cd remorph-openenv-submission && python scripts/export_frontend_payload.py
!cd remorph-openenv-submission && python scripts/promote_long_run_outputs.py || true

## 5) Export package + optional HF Jobs trigger from Colab

- Zip all submission artifacts for download/share.
- Optional: launch HF Jobs from Colab for bigger hardware runs (A10G/A100) when needed.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission
zip -rq /content/remorph_submission_artifacts.zip artifacts/submission
ls -lh /content/remorph_submission_artifacts.zip

echo "If you want to run on HF Jobs from Colab, use:"
echo "pip install -q -U huggingface_hub"
echo "hf jobs run --flavor a10g-large --secrets HF_TOKEN python:3.12 -- bash -lc '...your command...'"